In [1]:
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, classification_report

warnings.filterwarnings('ignore')
RANDOM_SEED = 42

SEARCH_PATHS = [
    (Path.cwd().parent / "data" / "flows.csv"),
    Path("mirai_flows.csv"),
    Path("Mirai.csv"),
    Path.home() / "Downloads" / "mirai_flows.csv",
    Path.home() / "datasets" / "mirai_flows.csv",
]

CSV_PATH = next((p for p in SEARCH_PATHS if p.exists()), None)
if CSV_PATH is None:
    raise FileNotFoundError(
        "Dataset Mirai no encontrado.\n"
        "Coloca el CSV con features de flujo en el directorio actual o en ~/Downloads/.\n"
        "Columna objetivo esperada: 'label' con valores 'benign' / nombre_ataque."
    )

df = pd.read_csv(CSV_PATH)
print(f"Shape: {df.shape}")
print(df['label'].value_counts())

Shape: (1006776, 13)
label
1    998983
0      7793
Name: count, dtype: int64


In [2]:
df = pd.read_csv(CSV_PATH)
df['src_ip'] = df['src_ip'].astype(str)

# Estado reclasificado semánticamente (igual que Markov Mirai)
def reclassify_state(row):
    proto  = row['state']
    b_pkts = row['b_pkts']
    avg_ps = row['avg_pkt_size']
    if proto == 'DNS':               return 'DNS_FLOOD'
    if proto in ('HTTP', 'HTTPS'):   return 'HTTP_FLOOD'
    if proto == 'SSH':               return 'OTHER'
    if proto == 'UDP_OTHER':
        if b_pkts == 0 and avg_ps < 100: return 'UDP_SMALL_NORESPONSE'
        elif b_pkts == 0:                return 'UDP_LARGE_NORESPONSE'
        else:                            return 'UDP_BIDIRECTIONAL'
    if proto == 'TCP_OTHER':
        if b_pkts == 0 and avg_ps < 80: return 'TCP_SYN_LIKE'
        elif b_pkts == 0:               return 'TCP_ACK_LIKE'
        else:                           return 'TCP_ESTABLISHED'
    return 'OTHER'
df['state'] = df.apply(reclassify_state, axis=1)

# Subsampleo por clase — igual que Markov Mirai (Hwang Tabla 12)
HWANG_CLASS_TOTAL = {
    'Normal':       68200 + 8525,
    'ACK_Flood':     6600 +  825,
    'DNS_Flood':     4312 +  539,
    'Mirai_CnC':    68200 + 8525,
    'GREIP_Flood':  24712 + 3089,
    'HTTP_Flood':     120 +   15,
    'SYN_Flood':    68200 + 8525,
    'UDP_Flood':    28816 + 3062,
    'VSE_Flood':     4432 +  554,
}

np.random.seed(42)
idx_keep_list = []
for cls_name, n_total in HWANG_CLASS_TOTAL.items():
    idx_cls = np.where(df['class_name'].values == cls_name)[0]
    if len(idx_cls) == 0:
        print(f'  AVISO: clase {cls_name!r} no encontrada')
        continue
    if len(idx_cls) > n_total:
        idx_cls = np.random.choice(idx_cls, n_total, replace=False)
    idx_keep_list.append(idx_cls)
idx_keep = np.sort(np.concatenate(idx_keep_list))
df = df.iloc[idx_keep].reset_index(drop=True)

# Features (igual que Markov Mirai)
FEATURE_COLS = ['n_pkts', 'n_bytes', 'f_pkts', 'f_bytes',
                'b_pkts', 'b_bytes', 'avg_pkt_size', 'duration', 'state']
df['state'] = LabelEncoder().fit_transform(df['state'].astype(str))

X = df[FEATURE_COLS].values.astype(np.float32)
y = df['label'].values.astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_SEED, stratify=y
)

# MinMaxScaler aplicado DESPUÉS del split (igual que Autoencoder Mirai)
scaler  = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print(f"Train: {len(X_train):,} | Test: {len(X_test):,}")
print(f"Features: {len(FEATURE_COLS)}")
print(f"Normal (0): {(y==0).sum():,}  |  Ataque (1): {(y==1).sum():,}")

Train: 129,107 | Test: 32,277
Features: 9
Normal (0): 7,793  |  Ataque (1): 153,591


In [3]:
CLASSIFIERS = {
    'KNN'     : KNeighborsClassifier(metric='euclidean'),
    'LDA'     : LinearDiscriminantAnalysis(),
    'DT'      : DecisionTreeClassifier(random_state=RANDOM_SEED),
    'RF'      : RandomForestClassifier(random_state=RANDOM_SEED),
    'LR'      : LogisticRegression(max_iter=1000, random_state=RANDOM_SEED),
    'SVM'     : SVC(kernel='rbf', gamma=0.1, random_state=RANDOM_SEED),
    'ANN'     : MLPClassifier(activation='logistic', solver='sgd', max_iter=500, random_state=RANDOM_SEED),
    'AdaBoost': AdaBoostClassifier(
                    estimator=DecisionTreeClassifier(max_depth=3, random_state=RANDOM_SEED),
                    n_estimators=100, random_state=RANDOM_SEED, algorithm='SAMME'
                ),
}

predictions = {}
for name, clf in CLASSIFIERS.items():
    print(f"Entrenando {name}...", end=' ', flush=True)
    clf.fit(X_train, y_train)
    predictions[name] = clf.predict(X_test)
    print("listo.")

Entrenando KNN... listo.
Entrenando LDA... listo.
Entrenando DT... listo.
Entrenando RF... listo.
Entrenando LR... listo.
Entrenando SVM... listo.
Entrenando ANN... listo.
Entrenando AdaBoost... listo.


In [4]:
SEP = '─' * 58
print("Clasificadores — Dataset Mirai")
print(SEP)
print(f"{'Modelo':<10} {'Prec':>6} {'Rec':>6} {'F1':>6} {'Acc':>6}")
print(f"{'─'*10} {'─'*6} {'─'*6} {'─'*6} {'─'*6}")

for name, y_pred in predictions.items():
    prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    rec  = recall_score   (y_test, y_pred, average='weighted', zero_division=0)
    f1   = f1_score       (y_test, y_pred, average='weighted', zero_division=0)
    acc  = accuracy_score (y_test, y_pred)
    print(f"{name:<10} {prec:.3f}  {rec:.3f}  {f1:.3f}  {acc:.3f}")

print(SEP)
print(classification_report(y_test, predictions['AdaBoost'],
                            target_names=['Benign', 'Malicious'], zero_division=0))

Clasificadores — Dataset Mirai
──────────────────────────────────────────────────────────
Modelo       Prec    Rec     F1    Acc
────────── ────── ────── ────── ──────
KNN        0.998  0.998  0.998  0.998
LDA        0.959  0.964  0.959  0.964
DT         0.998  0.998  0.998  0.998
RF         0.998  0.998  0.998  0.998
LR         0.957  0.957  0.941  0.957
SVM        0.969  0.969  0.963  0.969
ANN        0.952  0.954  0.934  0.954
AdaBoost   0.998  0.998  0.998  0.998
──────────────────────────────────────────────────────────
              precision    recall  f1-score   support

      Benign       0.98      0.99      0.98      1559
   Malicious       1.00      1.00      1.00     30718

    accuracy                           1.00     32277
   macro avg       0.99      0.99      0.99     32277
weighted avg       1.00      1.00      1.00     32277

